In [96]:
with open("names.txt", "r") as files:
    words = files.read().splitlines()

In [97]:
count: dict[tuple[str, str, str], int] = {}
for w in words:
    word = [".", "."] + list(w) + ["."]
    for ch1, ch2, ch3 in zip(word, word[1:], word[2:]):
        trigram = (ch1, ch2, ch3)
        count[trigram] = count.get(trigram, 0) + 1
sorted(count.items(), key=lambda kv: -kv[1])

[(('.', '.', 'a'), 4410),
 (('.', '.', 'k'), 2963),
 (('.', '.', 'm'), 2538),
 (('.', '.', 'j'), 2422),
 (('.', '.', 's'), 2055),
 (('a', 'h', '.'), 1714),
 (('.', '.', 'd'), 1690),
 (('n', 'a', '.'), 1673),
 (('.', '.', 'r'), 1639),
 (('.', '.', 'l'), 1572),
 (('.', '.', 'c'), 1542),
 (('.', '.', 'e'), 1531),
 (('a', 'n', '.'), 1509),
 (('o', 'n', '.'), 1503),
 (('.', 'm', 'a'), 1453),
 (('.', '.', 't'), 1308),
 (('.', '.', 'b'), 1306),
 (('.', 'j', 'a'), 1255),
 (('.', 'k', 'a'), 1254),
 (('e', 'n', '.'), 1217),
 (('.', '.', 'n'), 1146),
 (('l', 'y', 'n'), 976),
 (('y', 'n', '.'), 953),
 (('a', 'r', 'i'), 950),
 (('.', '.', 'z'), 929),
 (('i', 'a', '.'), 903),
 (('.', '.', 'h'), 874),
 (('i', 'e', '.'), 858),
 (('a', 'n', 'n'), 825),
 (('e', 'l', 'l'), 822),
 (('a', 'n', 'a'), 804),
 (('i', 'a', 'n'), 790),
 (('m', 'a', 'r'), 776),
 (('i', 'n', '.'), 766),
 (('e', 'l', '.'), 727),
 (('y', 'a', '.'), 716),
 (('a', 'n', 'i'), 703),
 (('.', 'd', 'a'), 700),
 (('l', 'a', '.'), 684),
 (('

In [98]:
chars = sorted(set("".join(words)))
stoi = {ch: i + 1 for i, ch in enumerate(chars)}
stoi["."] = 0
itos = {i: ch for ch, i in stoi.items()}

In [99]:
import torch

N = torch.zeros((27, 27, 27), dtype=torch.int32)
for w in words:
    word = [".", "."] + list(w) + ["."]
    for ch0, ch1, ch2 in zip(word, word[1:], word[2:]):
        id0 = stoi.get(ch0)
        id1 = stoi.get(ch1)
        id2 = stoi.get(ch2)
        N[id0, id1, id2] += 1

In [100]:
P = (N + 1).float()
P /= P.sum(2, keepdim=True)

g = torch.Generator().manual_seed(2147483647)
for i in range(100):
    out = []
    id0 = 0
    id1 = 0
    while True:
        p = P[id0, id1]
        id2 = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[id2])
        if id2 == 0:
            break
        id0 = id1
        id1 = id2
    print("".join(out))


ce.
bra.
jalius.
rochityharlonimittain.
luwak.
ka.
da.
samiyah.
javer.
gotai.
moriellavojkwuthda.
kaley.
maside.
en.
aviyah.
fobspihiliven.
tahlasuzusfxx.
an.
glhpynn.
isan.
jaridynne.
zam.
der.
jair.
tagaikayshaabelarl.
khysteeven.
abricayharien.
xmvpfkmwmghavaysor.
myson.
laitjaimilaydriseriyen.
kyille.
lahmie.
marah.
ammhgamaxemmy.
asharle.
alcalhy.
jayceasvz.
selane.
nellay.
ra.
adaliyana.
isa.
dougnichrishycero.
all.
jonn.
utstorgious.
ayra.
bekarie.
vikace.
ara.
jayk.
jagh.
crylesterlrklmace.
ro.
prah.
ye.
en.
aidgosell.
mer.
decla.
tie.
khamedahzymareizaymzjvtjuliah.
brik.
alkvsmjamere.
morad.
lie.
mariannanf.
mile.
keonteahaj.
ka.
rena.
mon.
keika.
suynn.
miciawathazhan.
jon.
stie.
kenie.
zakiha.
denovi.
kar.
kas.
try.
azemir.
ret.
ta.
ley.
ke.
sa.
carlorcagatai.
versimikavallin.
skyohaen.
caikeyaderaydew.
aarrecher.
ole.
prisy.
elyn.
jassyn.
abrey.
oah.


In [101]:
xs: list[tuple[int, int]] = []
ys: list[int] = []
for w in words:
    word = [".", "."] + list(w) + ["."]
    for ch0, ch1, ch2 in zip(word, word[1:], word[2:]):
        id0 = stoi[ch0]
        id1 = stoi[ch1]
        id2 = stoi[ch2]
        xs.append((id0, id1))
        ys.append(id2)
xs = torch.tensor([i * 27 + j for i, j in xs])
ys = torch.tensor(ys)
num = xs.nelement()

In [102]:
import torch.nn.functional as F

xenc = F.one_hot(xs, num_classes=27 * 27).float()

In [103]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27 * 27, 27), generator=g, requires_grad=True)

steps = 200
for i in range(steps):
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)
    sump = -probs[torch.arange(num), ys].log().mean()
    alpha = 0.01
    loss = sump + alpha * (W ** 2).mean()

    W.grad = None
    loss.backward()

    step = 100 - (100 - 1) * 1.0 / steps * i 
    W.data += -step * W.grad

    print(f"Step {i}, step size {step}, {loss.data=}")

Step 0, step size 100.0, loss.data=tensor(3.8028)
Step 1, step size 99.505, loss.data=tensor(3.5481)
Step 2, step size 99.01, loss.data=tensor(3.4281)
Step 3, step size 98.515, loss.data=tensor(3.3367)
Step 4, step size 98.02, loss.data=tensor(3.2607)
Step 5, step size 97.525, loss.data=tensor(3.1966)
Step 6, step size 97.03, loss.data=tensor(3.1417)
Step 7, step size 96.535, loss.data=tensor(3.0940)
Step 8, step size 96.04, loss.data=tensor(3.0519)
Step 9, step size 95.545, loss.data=tensor(3.0143)
Step 10, step size 95.05, loss.data=tensor(2.9804)
Step 11, step size 94.555, loss.data=tensor(2.9498)
Step 12, step size 94.06, loss.data=tensor(2.9219)
Step 13, step size 93.565, loss.data=tensor(2.8964)
Step 14, step size 93.07, loss.data=tensor(2.8730)
Step 15, step size 92.575, loss.data=tensor(2.8515)
Step 16, step size 92.08, loss.data=tensor(2.8317)
Step 17, step size 91.58500000000001, loss.data=tensor(2.8134)
Step 18, step size 91.09, loss.data=tensor(2.7964)
Step 19, step size 90

In [104]:
W


tensor([[-3.1946,  2.1436,  0.9263,  ..., -1.3536,  0.0331,  0.5854],
        [-1.9723,  0.6786,  0.5910,  ..., -1.1361,  0.4946,  0.3571],
        [-0.5396,  2.5802, -0.4312,  ..., -0.9962, -0.4332, -1.3884],
        ...,
        [-0.5498,  0.3982, -0.3281,  ..., -0.5030, -2.0144, -0.1574],
        [ 1.3222,  0.0220, -1.3237,  ..., -1.1063,  0.1783, -1.0482],
        [ 0.2235,  0.5597,  0.4898,  ...,  1.1092,  0.0970,  1.2508]],
       requires_grad=True)

In [105]:
for i in range(30):
    out = []
    id0 = 0
    id1 = 0
    while True:
        context_enc = F.one_hot(torch.tensor([id0 * 27 + id1]), num_classes=27 * 27).float()
        logits = context_enc @ W
        counts = logits.exp()
        probs = counts / counts.sum(1, keepdim=True)

        id2 = torch.multinomial(probs, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[id2])
        if id2 == 0:
            break
        id0 = id1
        id1 = id2
    print("".join(out))

dah.
ree.
jueolizeniqwzyijkycnziaray.
milytnzola.
mah.
bpznleyan.
chsnie.
clrkey.
cykolueana.
bryus.
jyverri.
bra.
te.
kajuqpifaxsuriellon.
raylina.
an.
hygamardfa.
korlkyleish.
aminchimdsmazidxxskaackwlla.
earijnqwybwvquchagvhisqzlyn.
ken.
yxckezsifjayseri.
braymadin.
rouw.
khaltren.
jyvvgoxrbjiuyverelia.
kydzuvgxptznrlhaens.
aeeqiuyye.
lbakasha.
emiragyf.
